# MP28 — Multi-Head Self-Attention

Build feature projections, split heads, concatenate outputs and apply the output projection. [Brief](../stage_07_transformers/MP28_multi_head_self_attention/README.md) · [Validation](../stage_07_transformers/MP28_multi_head_self_attention/VALIDATION.md).

Prerequisite: MP27's checked single-head attention. No training, network or held-out evaluation. This is a bias-free NumPy float64 CPU demonstration, not a complete Transformer.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import matplotlib.pyplot as plt
root=Path.cwd() if (Path.cwd()/'src/rssr').exists() else Path.cwd().parent
sys.path.insert(0,str(root/'stage_07_transformers/MP28_multi_head_self_attention'))
from multihead_tools import multihead,projection_parameters
from attention_tools import attention


## 1. Follow the dimensions

For X[N,D], four projection matrices Wq,Wk,Wv,Wo each have D×D entries. With h heads, require D divisible by h and let d=D/h. Split XWq, XWk, XWv along their feature columns into h arrays[N,d]. Each head creates a probability matrix[N,N] using √d scaling and produces[N,d]. Concatenating gives[N,D], then multiplying Wo keeps[N,D].

For N=3,D=4,h=2: each head has width2, maps stack as[2,3,3], final output[3,4]. Bias-free attention uses4D²=64 scalar projection entries, independent of h when D stays fixed. More heads do not automatically mean more parameters; retaining width per head instead of total width would be a different experiment.

In [ ]:
x=np.arange(12,dtype=float).reshape(3,4)/12
y,maps=multihead(x,[np.eye(4)]*4,2)
assert y.shape==(3,4) and maps.shape==(2,3,3)
assert projection_parameters(4)==64
print('output',y.shape,'attention',maps.shape,'parameters',projection_parameters(4))


## 2. Two scalar heads by hand

Let X be the2×2 identity and all projections identity. Each head has width1, so its divisor is1. Head1 sees token features[1,0]: attention rows[p,1−p] and[0.5,0.5], where p=e/(e+1)=0.731059. Head2 sees[0,1] and reverses the pattern. The concatenated output is[[p,0.5],[0.5,p]].

In [ ]:
p=np.exp(1)/(np.exp(1)+1)
y,maps=multihead(np.eye(2),[np.eye(2)]*4,2)
np.testing.assert_allclose(y,[[p,.5],[.5,p]])
print('Hand-checked output:',y)
fig,axes=plt.subplots(1,2,figsize=(7,3))
for i,ax in enumerate(axes):
    ax.imshow(maps[i],vmin=0,vmax=1,cmap='viridis')
    for a in range(2):
        for b in range(2):ax.text(b,a,f'{maps[i,a,b]:.3f}',ha='center',color='white')
    ax.set(title=f'Head {i+1}',xlabel='Key token',ylabel='Query token',xticks=[0,1],yticks=[0,1])
fig.tight_layout();plt.show()


## 3. Controlled one/two/four-head experiment

Keep total width4, tokens, projection matrices and seed28 fixed; change only the head partition. Use seeded Gaussian tokens first, then16 RGB pixel tokens from a4×4 valid NAIP training crop embedded by a fixed3×4 random matrix. The embedding is untrained and adds12 entries outside the64 attention projection entries. This experiment does not test semantic specialization.

The next cell reads previously measured evidence, so viewing the notebook does not repeat acquisition or experiments.

In [ ]:
record=json.loads((root/'reports/mp28-multihead.json').read_text())
assert record['source']['split']=='train'
assert all(r['finite'] and r['max_row_sum_error']<1e-12 for r in record['rows'])
for row in record['rows']:
    print(row['input'],row['heads'],'heads; shape',row['attention_shape'],
          'parameters',row['projection_parameters'],'entropy',row['head_entropy_nats'])


## 4. Inspect head maps on the same color scale

Compare the Gaussian fixture's maps below. All axes use0–1 probability. Different random projections/subspaces can yield different maps without learning anything useful. The real crop can remain almost uniform because its normalized values are small. Neither pattern establishes SR quality.

In [ ]:
rows=[r for r in record['rows'] if r['input']=='synthetic_normal']
fig,axes=plt.subplots(3,4,figsize=(10,8),squeeze=False)
for row_index,row in enumerate(rows):
    for head,ax in enumerate(axes[row_index]):
        if head>=row['heads']:
            ax.axis('off');continue
        im=ax.imshow(row['maps'][head],vmin=0,vmax=1,cmap='viridis')
        ax.set(title=f"{row['heads']} heads / head {head+1}",xlabel='Key token',ylabel='Query token')
fig.tight_layout();plt.show()


## 5. Parameters versus intermediate storage

For16 tokens each float64 attention map stores16×16×8=2048 bytes. At1,2,4 heads that is2048,4096,8192 bytes even though the projection count remains64. These measured array nbytes exclude other arrays, temporaries and gradients; they are not peak memory. A larger head count alone is not a quality improvement.

In [ ]:
for row in record['rows']:
    assert row['projection_parameters']==64
    assert row['float64_attention_bytes']==row['heads']*16*16*8
print('Parameter and map-storage checks passed')


## Practice before solutions

1. What fails for D=4,h=3? 2. If Wo is zero, what is the output? 3. Does concatenation add token positions? 4. Why is comparing fixed head width across head counts confounded?

## Solutions, reflection and limits

1. Equal-width heads cannot partition four features into three groups. 2. Output is zero regardless of the maps. 3. No: concatenation joins features and has no coordinate information. 4. Total width and parameter count change, so this is not a head-count-only comparison.

Learned: head splitting preserves output shape but changes the feature comparisons. Failure case: incompatible dimensions must be rejected. Next: residual connections, normalization and a feed-forward block in MP29. No mask, positional encoding, dropout or learned task is included here. Learner mastery is unassessed.

Source: NAIP public-domain aerial imagery with USDA/OpenSR attribution; dataset-card MIT; stored DN/10000, physical calibration unverified. The measured report pins the source SHA and crop. No frozen test or validation sample is used.

Reference: Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), multi-head attention. Only the attention primitive is implemented.

In [ ]:
try:
    multihead(np.eye(4),[np.eye(4)]*4,3)
except ValueError as error:print('Expected rejection:',error)
else:raise AssertionError('Invalid head count accepted')
out,_=multihead(np.eye(4),[np.eye(4)]*3+[np.zeros((4,4))],2)
np.testing.assert_array_equal(out,np.zeros((4,4)))
